# 로컬 판단모델 vs GPT-4o-mini 평가 (v3 — 정답을 GPT-4o-mini로 생성)

3개 모델을 4개 실제 프로덕션 판단 태스크로 비교합니다: `gpt-4o-mini`(API),
`Qwen2.5-7B-Instruct-bnb-4bit`(텍스트 전용), `Qwen2.5-VL-7B-Instruct-bnb-4bit`
(비전 모델, 텍스트만 입력해서 사용). 두 로컬 모델 모두 4bit로 이미
양자화된 체크포인트를 그대로 쓰므로 양자화 기준이 동일합니다.

**태스크 4개와 정답(ground truth) 정하는 방식**

| 태스크 | 입력 | 정답 출처 |
|---|---|---|
| `item_spec_validation` | item_group + 실제 서술형 설명 텍스트 + 필수규격 목록 | 사람이 설명 텍스트를 직접 써서 참/거짓이 명확함 (직접 작성) |
| `item_spec_definition` | item_group 이름만 | **gpt-4o-mini가 그 자리에서 생성한 답을 정답으로 사용** |
| `find_substitute` | 품목명·설명·필요수량·후보 목록(재고 포함) | **gpt-4o-mini가 그 자리에서 생성한 답을 정답으로 사용** |
| `sq_evaluation` | RFQ 요구사항 + 여러 견적(단가·수량·납기·규격설명) | **gpt-4o-mini가 그 자리에서 생성한 답을 정답으로 사용** |

gpt-4o-mini가 정답 생성자 역할과 평가 대상 역할을 동시에 하기 때문에,
`gpt-4o-mini` 자신의 점수는 자기 자신과 비교한 값이라 1.0에 가깝게
나오는 게 정상입니다 — 만약 1.0에서 많이 벗어나면 API 비결정성이나
채점 코드 버그를 먼저 의심하세요(sanity check).

**참고**: `item_spec_validation`(완결성 체크)의 실제 프로덕션 프롬프트는
"필수규격 항목에 값이 실제로 채워져 있는지"만 판단하고, 그 값이 해당
항목에 의미상 맞는 타입인지(예: '사이즈' 항목에 색상값이 들어간 경우)는
검증하지 않습니다 — 소스코드로 확인했습니다. 그래서 이번 테스트케이스에는
그런 시나리오를 넣지 않았습니다.


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece openai huggingface_hub

In [ ]:
import os, json, re, time, gc
from getpass import getpass

# 로컬(Colab 디스크) 캐시만 사용 - Google Drive 캐시는 무료용량(15GB) 초과
# 문제가 반복돼서 이번 버전에서는 아예 안 씀. 런타임 끊기면 다시 받아야
# 하지만 unsloth 4bit 체크포인트가 각 4~5GB라 재다운로드 부담이 적음.
os.environ["HF_HOME"] = "/content/hf_cache"

OPENAI_API_KEY = getpass("OpenAI API 키 입력: ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

TEXT_MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
VISION_MODEL_ID = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
OPENAI_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 768


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
    AutoProcessor,
)

def load_text_model():
    tok = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        TEXT_MODEL_ID,
        device_map={"": 0},   # "auto"가 CPU/disk로 오프로드하려던 문제 회피 - GPU 하나에 강제 배치
        torch_dtype=torch.bfloat16,
    )
    model.eval()
    return tok, model

def load_vision_model_text_only():
    proc = AutoProcessor.from_pretrained(VISION_MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        VISION_MODEL_ID,
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
    )
    model.eval()
    return proc, model

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
from openai import OpenAI

_openai_client = OpenAI(api_key=OPENAI_API_KEY)

def run_gpt(prompt: str):
    t0 = time.time()
    resp = _openai_client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed = time.time() - t0
    return resp.choices[0].message.content, elapsed


def run_hf_text(tok, model, prompt: str):
    messages = [{"role": "user", "content": prompt}]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    elapsed = time.time() - t0
    text = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text, elapsed


def run_hf_vision_text_only(proc, model, prompt: str):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = proc.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    elapsed = time.time() - t0
    text = proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text, elapsed


In [ ]:
def parse_json_response(text: str) -> dict:
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError(f"JSON을 못 찾음: {text[:200]!r}")
        parsed = json.loads(match.group(0))
    if not isinstance(parsed, dict):
        raise ValueError(f"최상위가 dict가 아님 ({type(parsed).__name__}): {str(parsed)[:200]!r}")
    return parsed


In [ ]:
def _strip_html(text):
    if not text:
        return ""
    plain = re.sub(r"<[^>]+>", " ", text)
    plain = re.sub(r"\s+", " ", plain)
    return plain.strip()


# --- item_spec_validation.py : _ai_define_required_specs ---
def build_prompt_item_spec_definition(item_group: str) -> str:
    return (
        "당신은 기업 구매팀의 품목등록 검수 담당자입니다. "
        "다음 품목분류(item_group)에 대해, 구매요청서에 \'이것만은 반드시\' "
        "적혀 있어야 하는 최소 필수 규격 항목을 정하세요.\n\n"
        f"품목분류: {item_group}\n\n"
        "규칙:\n"
        "- 정말 이게 없으면 발주 자체가 불가능한 항목만 최소로 고르세요 "
        "(과하게 많이 요구하지 마세요, 통상 2~5개 사이).\n"
        "- 항목명은 짧은 한국어 명사로 (예: \'재질\', \'규격(치수)\', \'전압\', \'용량\').\n"
        "- 왜 이 항목들을 필수로 골랐는지 한두 문장으로 이유를 남기세요.\n\n"
        '반드시 이 JSON 형식으로만 답하세요: '
        '{"required_specs": ["항목1", "항목2"], "reason": "짧은 이유"}'
    )


# --- item_spec_validation.py : _ai_check_completeness ---
def build_prompt_item_spec_validation(item_group: str, description: str, required_specs: list) -> str:
    required_specs_str = ", ".join(required_specs)
    return (
        f"다음은 품목분류 \'{item_group}\'에 대한 구매요청 설명입니다.\n\n"
        f"[설명]\n{description}\n\n"
        f"[확인해야 할 필수규격 목록]\n{required_specs_str}\n\n"
        "각 필수규격 항목이 이 설명에 실제 값까지 구체적으로 채워져서 "
        "기재됐는지 판단하세요. 항목 이름(라벨)만 있고 값이 비어있거나, "
        "아예 언급이 없으면 \'미기재\'입니다. 라벨이 명시적으로 없어도 "
        "설명 안에 그 값이 자연스럽게 들어있으면 \'기재됨\'으로 인정하세요.\n\n"
        '반드시 이 JSON 형식으로만 답하세요: '
        '{"results": [{"spec": "항목명", "present": true, "reason": "짧은 이유"}]}'
    )


# --- find_substitute.py : _ai_rank_substitutes ---
def build_prompt_find_substitute(item_name, item_description, qty_needed, candidates, max_results=5) -> str:
    candidates_json = [
        {
            "item_code": c["item_code"],
            "item_name": c["item_name"],
            "description": _strip_html(c.get("description") or ""),
            "재고": c["total_qty"],
        }
        for c in candidates
    ]
    candidates_str = json.dumps(candidates_json, ensure_ascii=False, indent=2)
    return (
        f"요청품목: {item_name}\n"
        f"요청품목 설명: {_strip_html(item_description)}\n"
        f"요청수량: {qty_needed}\n\n"
        "아래 후보 품목들 중에서, 실제로 요청품목을 대신 쓸 수 있는 것들만 "
        f"골라서 적합도 순으로 순위를 매겨주세요 (최대 {max_results}개).\n\n"
        f"후보 목록:\n{candidates_str}\n\n"
        "규칙:\n"
        "- 용도·사용대상이 명확히 다른 물건(예: 화이트보드용 vs 연필용 지우개)은 "
        "이름이 비슷해도 제외하세요.\n"
        "- 스펙이 원본보다 낮아도(다운그레이드) 용도가 같으면 후보에 포함하되, "
        "reason에 그 사실을 명시하세요.\n"
        "- 재고가 요청수량보다 적으면 reason에 \'부분충족(N개)\'이라고 언급하세요.\n"
        "- 적합한 후보가 하나도 없으면 빈 리스트를 반환하세요.\n\n"
        '반드시 이 JSON 형식으로만 답하세요: '
        '{"ranking": [{"item_code": "...", "rank": 1, "reason": "짧은 이유"}]}'
    )


# --- sq_evaluation.py : _ai_rank_quotations ---
def build_prompt_sq_evaluation(requirements: dict, quotations: list) -> str:
    requirements_str = json.dumps(requirements, ensure_ascii=False, indent=2, default=str)
    quotations_str = json.dumps(quotations, ensure_ascii=False, indent=2, default=str)
    return (
        "다음은 RFQ 요청내용과, 여기에 대해 여러 공급사가 제출한 견적입니다.\n\n"
        f"[RFQ 요청내용]\n{requirements_str}\n\n"
        f"[제출된 견적들]\n{quotations_str}\n\n"
        "각 견적을 요청내용과 대조해서 판단하세요:\n\n"
        "1. 품목 매칭 및 규격 비교 (가장 중요):\n"
        "   - item_code가 일치하는지 확인하되, item_code만으로 판단하지 "
        "마세요. 각 요청 품목의 description(사양)과 견적 품목의 "
        "description을 실제 내용으로 대조하세요.\n"
        "   - item_code가 같아도 description에 나온 재질,규격,등급,"
        "치수 등이 요청과 다르면 \'규격 불일치\'로 issues에 명시하세요.\n"
        "   - item_code가 비어있거나 다르더라도, description 내용이 "
        "요청 품목과 실질적으로 동일한 것을 가리키면 매칭된 것으로 "
        "간주하고 그 사실을 reason에 명시하세요.\n"
        "   - 요청보다 낮은 사양(다운그레이드)이면 issues에 구체적으로 "
        "어떤 사양이 부족한지 적으세요.\n\n"
        "2. 수량 충족: 요청수량보다 적게 제출한 견적은 \'부분충족\'으로 "
        "명시하되 순위에는 포함하세요 (완전배제는 하지 마세요, 담당자가 "
        "판단할 수 있게).\n\n"
        "3. 순위: 규격/수량을 충족하는 것들만 가격(낮을수록 좋음)과 "
        "납기(빠를수록 좋음)를 기준으로 순위를 매기세요. 규격이 명확히 "
        "다르거나 요청 품목 자체가 빠진 견적은 순위 최하위로 두고 "
        "issues에 사유를 남기세요.\n\n"
        "reason은 한두 문장으로 짧게, 왜 이 순위인지(규격 일치여부 포함)를 "
        "포함하세요.\n\n"
        '반드시 이 JSON 형식으로만 답하세요: {"ranking": [{"name": "견적문서명", '
        '"supplier": "공급사명", "rank": 1, "fulfills_qty": true, '
        '"spec_match": true, "reason": "짧은 이유", "issues": []}]}'
    )


# --- 이번 v3에서 새로 추가: item_spec_definition 채점용 LLM 심사 프롬프트 ---
def build_prompt_judge_spec_definition(item_group, reference_specs, candidate_specs) -> str:
    ref_str = ", ".join(reference_specs) if reference_specs else "(빈 목록)"
    cand_str = ", ".join(candidate_specs) if candidate_specs else "(빈 목록)"
    return (
        "당신은 구매팀 규격관리 감사자입니다. 아래는 특정 품목분류에 대해 "
        "\'반드시 필요한 최소 규격 항목\'을 두 명이 각각 정의한 목록입니다.\n\n"
        f"품목분류: {item_group}\n\n"
        f"[기준 목록(정답으로 간주)]\n{ref_str}\n\n"
        f"[비교 대상 목록]\n{cand_str}\n\n"
        "두 목록의 항목명이 문자열은 달라도 의미가 같으면(예: \'재질\'과 "
        "\'소재\', \'규격\'과 \'규격(치수)\') 같은 항목으로 간주하세요. 아래를 "
        "판단하세요:\n"
        "1. 기준 목록의 각 항목에 대해, 비교 대상 목록에 의미상 대응하는 "
        "항목이 있는지\n"
        "2. 비교 대상 목록에만 있고 기준 목록에는 의미상 대응 항목이 없는 "
        "불필요한 항목이 있는지\n\n"
        '반드시 이 JSON 형식으로만 답하세요: {"matched_reference_items": '
        '["기준 목록 중 대응 항목을 찾은 것들 (기준 목록 표현 그대로)"], '
        '"unmatched_reference_items": ["대응 항목을 못 찾은 기준 목록 항목들"], '
        '"extra_candidate_items": ["기준 목록에 없는데 비교 대상에만 있는 항목들"]}'
    )


In [ ]:
TEST_CASES = {
    "item_spec_validation": [],
    "item_spec_definition": [],
    "find_substitute": [],
    "sq_evaluation": [],
}

def add_validation(item_group, description, required_specs, present_map):
    TEST_CASES["item_spec_validation"].append({
        "input": {"item_group": item_group, "description": description, "required_specs": required_specs},
        "expected": {"results": [{"spec": s, "present": present_map[s]} for s in required_specs]},
    })

# label 명시 + 값 다 채워짐
add_validation(
    "안전모", "재질: HDPE, 색상: 백색, 인증등급 KCs 인증 획득",
    ["재질", "색상", "인증등급"], {"재질": True, "색상": True, "인증등급": True},
)
# label 없이 자연스럽게 값이 문장에 녹아있음
add_validation(
    "장갑", "니트릴 코팅장갑, Cut Level D, 사이즈 L",
    ["소재", "사이즈", "절단등급"], {"소재": True, "사이즈": True, "절단등급": True},
)
# 아예 언급 없음
add_validation(
    "소화기", "소화기 5개 구매 요청, 사무실 배치용입니다",
    ["용량", "타입", "압력방식"], {"용량": False, "타입": False, "압력방식": False},
)
# label은 있는데 값이 비어있는 항목 하나 섞임 (제일 까다로운 케이스)
add_validation(
    "전선", "전압: , 굵기: 2.5SQ, 피복재질: PVC (전압은 담당자 확인 후 회신 예정)",
    ["전압", "굵기", "피복재질"], {"전압": False, "굵기": True, "피복재질": True},
)
# 일부만 언급된 혼합 케이스
add_validation(
    "밸브", "스테인리스 재질 밸브이며 10K 압력등급 제품으로 부탁드립니다",
    ["구경", "압력등급", "재질"], {"구경": False, "압력등급": True, "재질": True},
)
# "강화선심"만으로는 등급까지는 알 수 없는, 애매해서 헷갈리기 쉬운 케이스
add_validation(
    "안전화", "미끄럼방지 밑창에 강화선심 들어간 걸로 부탁드립니다. 사이즈는 다양하게 260~280 섞어서요",
    ["재질", "사이즈", "선심등급"], {"재질": False, "사이즈": True, "선심등급": False},
)
# "추후 안내"라고 명시적으로 미룬 케이스 - 전부 미기재여야 정상
add_validation(
    "케이블", "실외배선용 케이블 300미터 구매 예정, 규격은 추후 현장 실측 후 안내드리겠습니다",
    ["규격(굵기)", "전압", "심선수"], {"규격(굵기)": False, "전압": False, "심선수": False},
)
# label 없이 자연스럽게 다 채워진 케이스 (다른 품목군으로 한번 더 확인)
add_validation(
    "복사용지", "A4 사이즈, 평량 80g, 백색도 100 이상 제품으로 500박스 주문합니다",
    ["규격(사이즈)", "평량"], {"규격(사이즈)": True, "평량": True},
)

# item_spec_definition: item_group 이름만 입력, 정답은 이번엔 gpt-4o-mini가 그 자리에서 생성
for _ig in ["안전화", "케이블", "복사용지", "소화기", "방진마스크", "사무용 의자", "산업용 장갑", "절연테이프"]:
    TEST_CASES["item_spec_definition"].append({"input": {"item_group": _ig}})


def add_substitute(item_name, item_description, qty_needed, candidates):
    TEST_CASES["find_substitute"].append({
        "input": {
            "item_name": item_name, "item_description": item_description,
            "qty_needed": qty_needed, "candidates": candidates,
        },
    })

add_substitute(
    "안전모(백색)", "건설현장용 안전모", 10,
    [
        {"item_code": "IT-002", "item_name": "안전모(황색)", "description": "건설현장용 안전모, HDPE 재질, 동일 인증등급", "total_qty": 100},
        {"item_code": "IT-003", "item_name": "안전모(청색)", "description": "건설현장용 안전모, HDPE 재질, 동일 인증등급", "total_qty": 40},
        {"item_code": "IT-004", "item_name": "공사현장용 안전조끼", "description": "형광 안전조끼, 머리 보호 기능 없음", "total_qty": 200},
        {"item_code": "IT-005", "item_name": "자전거 헬멧", "description": "레저용 자전거 헬멧, 산업안전 인증 없음", "total_qty": 15},
    ],
)
add_substitute(
    "니트릴 코팅장갑 (Cut Level D)", "절단위험 작업용 안전장갑", 50,
    [
        {"item_code": "IT-010", "item_name": "니트릴 코팅장갑 (Cut Level C)", "description": "동일 브랜드, 절단등급만 한 단계 낮음", "total_qty": 80},
        {"item_code": "IT-011", "item_name": "라텍스 코팅장갑 (Cut Level D)", "description": "코팅 재질만 라텍스, 절단등급 동일", "total_qty": 60},
        {"item_code": "IT-012", "item_name": "니트릴 코팅장갑 (Cut Level E)", "description": "절단등급 한 단계 상위, 나머지 동일", "total_qty": 30},
        {"item_code": "IT-013", "item_name": "일반 면장갑", "description": "절단보호 기능 없는 일반 작업용 면장갑", "total_qty": 200},
    ],
)
add_substitute(
    "A4 복사용지 (80g)", "사무용 복사용지", 20,
    [
        {"item_code": "IT-030", "item_name": "A4 복사용지 (75g)", "description": "A4 사이즈, 평량만 75g로 낮음", "total_qty": 200},
        {"item_code": "IT-031", "item_name": "A4 복사용지 (90g)", "description": "A4 사이즈, 평량만 90g로 높음", "total_qty": 50},
        {"item_code": "IT-032", "item_name": "A3 복사용지 (80g)", "description": "A3 사이즈, 평량 동일", "total_qty": 100},
        {"item_code": "IT-033", "item_name": "A4 감열지", "description": "팩스/영수증 출력용 감열지, 일반 인쇄용 아님", "total_qty": 80},
    ],
)
add_substitute(
    "방진마스크 (N95)", "분진 작업용 보호구", 200,
    [
        {"item_code": "IT-060", "item_name": "방진마스크 (N99)", "description": "N95보다 상위등급, 방진성능 우수", "total_qty": 25},
        {"item_code": "IT-061", "item_name": "방진마스크 (N90)", "description": "N95보다 한 단계 낮은 등급, 동일 용도", "total_qty": 150},
        {"item_code": "IT-062", "item_name": "수술용 마스크", "description": "의료용 비말차단 마스크, 산업분진 차단 인증 없음", "total_qty": 500},
    ],
)
add_substitute(
    "CNC 가공 서비스 (알루미늄)", "알루미늄 부품 CNC 가공 외주", 1,
    [
        {"item_code": "IT-070", "item_name": "CNC 가공용 밀링머신", "description": "CNC 가공에 쓰이는 장비 자체(밀링머신) 판매", "total_qty": 5},
        {"item_code": "IT-071", "item_name": "아노다이징 표면처리 서비스", "description": "가공이 아니라 표면처리(도금) 서비스", "total_qty": 10},
        {"item_code": "IT-072", "item_name": "알루미늄 원자재 (AL6061 판재)", "description": "가공 안 된 원자재 판매, 서비스 아님", "total_qty": 500},
    ],
)
add_substitute(
    "사무용 의자 (메쉬, 팔걸이형)", "사무실 개인 책상용 의자", 5,
    [
        {"item_code": "IT-080", "item_name": "사무용 의자 (메쉬, 팔걸이 없음)", "description": "메쉬 등판 동일, 팔걸이만 없음", "total_qty": 20},
        {"item_code": "IT-081", "item_name": "사무용 의자 (가죽, 팔걸이형)", "description": "고급 가죽 등판, 팔걸이형", "total_qty": 10},
        {"item_code": "IT-082", "item_name": "회의실용 스툴", "description": "팔걸이·등받이 모두 없는 간이 스툴, 회의실 전용", "total_qty": 30},
        {"item_code": "IT-083", "item_name": "좌식 방석", "description": "바닥에 놓는 좌식용 방석, 의자 아님", "total_qty": 50},
    ],
)
add_substitute(
    "볼트&너트 M8x20 (스테인리스)", "기계조립용 체결부품", 500,
    [
        {"item_code": "IT-090", "item_name": "볼트&너트 M8x25 (스테인리스)", "description": "길이만 5mm 더 김, 재질/규격 동일", "total_qty": 300},
        {"item_code": "IT-091", "item_name": "볼트&너트 M8x20 (아연도금)", "description": "재질만 아연도금으로 다름, 저가형", "total_qty": 1000},
        {"item_code": "IT-092", "item_name": "볼트&너트 M10x20 (스테인리스)", "description": "굵기가 달라 호환 안 됨", "total_qty": 200},
        {"item_code": "IT-093", "item_name": "목재용 나사", "description": "목공용 나사, 금속 체결 용도 아님", "total_qty": 500},
    ],
)
add_substitute(
    "프린터 토너 (HP 26A 정품)", "사무실 레이저프린터용 소모품", 10,
    [
        {"item_code": "IT-100T", "item_name": "프린터 토너 (HP 26A 재생)", "description": "동일 규격 재생토너, 호환됨", "total_qty": 40},
        {"item_code": "IT-101T", "item_name": "프린터 토너 (HP 26X 고용량)", "description": "동일 프린터 기종 호환, 용량만 더 큼", "total_qty": 15},
        {"item_code": "IT-102T", "item_name": "프린터 토너 (Canon 정품)", "description": "다른 프린터 기종 전용, 호환 안 됨", "total_qty": 20},
        {"item_code": "IT-103T", "item_name": "A4 복사용지", "description": "프린터 소모품이 아닌 용지류", "total_qty": 100},
    ],
)


def add_sq(rfq_name, req_items, quotations):
    TEST_CASES["sq_evaluation"].append({
        "input": {"requirements": {"rfq_name": rfq_name, "items": req_items}, "quotations": quotations},
    })

add_sq(
    "RFQ-1001",
    [{"item_code": "IT-100", "item_name": "안전모", "description": "HDPE 재질, 백색, KCs 인증", "qty": 50}],
    [
        {"name": "SQ-1001", "supplier": "가나상사", "items": [{"item_code": "IT-100", "description": "HDPE 재질, 백색, KCs 인증", "qty": 50, "rate": 5000}]},
        {"name": "SQ-1002", "supplier": "다라상사", "items": [{"item_code": "IT-100", "description": "HDPE 재질, 백색, KCs 인증", "qty": 50, "rate": 4500}]},
        {"name": "SQ-1003", "supplier": "마바상사", "items": [{"item_code": "IT-100", "description": "PP 재질 저가형, 인증 미표기", "qty": 50, "rate": 4000}]},
    ],
)
add_sq(
    "RFQ-2001",
    [{"item_code": "IT-200", "item_name": "니트릴 코팅장갑", "description": "니트릴 코팅장갑 Cut Level D", "qty": 100}],
    [
        {"name": "SQ-2001", "supplier": "A산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 Cut Level D", "qty": 100, "rate": 3000}]},
        {"name": "SQ-2002", "supplier": "B산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 (등급 표기 없음)", "qty": 100, "rate": 2000}]},
        {"name": "SQ-2003", "supplier": "C산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 Cut Level C", "qty": 100, "rate": 2500}]},
    ],
)
add_sq(
    "RFQ-3001",
    [{"item_code": "IT-500", "item_name": "복사용지", "description": "A4 사이즈, 80g, 2500매/박스", "qty": 100}],
    [
        {"name": "SQ-3001", "supplier": "G문구", "items": [{"item_code": "", "description": "A4용지 80g 2500매/박스", "qty": 100, "rate": 22000}]},
        {"name": "SQ-3002", "supplier": "H문구", "items": [{"item_code": "IT-500", "description": "A4 재생용지 75g 2500매", "qty": 100, "rate": 20000}]},
        {"name": "SQ-3003", "supplier": "I문구", "items": [{"item_code": "IT-500", "description": "A4용지 80g 2500매/박스", "qty": 100, "rate": 25000}]},
    ],
)
add_sq(
    "RFQ-4001",
    [{"item_code": "IT-600", "item_name": "청소용 세제", "description": "동일 사양", "qty": 200}],
    [
        {"name": "SQ-4001", "supplier": "H상사", "items": [{"item_code": "IT-600", "description": "동일 사양", "qty": 200, "rate": 1000}]},
        {"name": "SQ-4002", "supplier": "I상사", "items": [{"item_code": "IT-600", "description": "동일 사양", "qty": 100, "rate": 850}]},
        {"name": "SQ-4003", "supplier": "J상사", "items": [{"item_code": "IT-600", "description": "동일 사양", "qty": 200, "rate": 950}]},
    ],
)
add_sq(
    "RFQ-5001",
    [{"item_code": "IT-700", "item_name": "밸브", "description": "스테인리스 재질, 10K 압력등급", "qty": 50}],
    [
        {"name": "SQ-5001", "supplier": "K상사", "items": [{"item_code": "IT-700", "description": "스테인리스 재질, 10K 압력등급", "qty": 50, "rate": 2000, "expected_delivery_date": "2026-09-20"}]},
        {"name": "SQ-5002", "supplier": "L상사", "items": [{"item_code": "IT-700", "description": "스테인리스 재질, 10K 압력등급", "qty": 50, "rate": 2000, "expected_delivery_date": "2026-09-10"}]},
        {"name": "SQ-5003", "supplier": "M상사", "items": [{"item_code": "IT-700", "description": "스테인리스 재질, 10K 압력등급", "qty": 50, "rate": 2000, "expected_delivery_date": "2026-09-25"}]},
    ],
)
add_sq(
    "RFQ-6001",
    [{"item_code": "IT-800", "item_name": "안전화", "description": "강화선심, 미끄럼방지 밑창, 사이즈 260", "qty": 30}],
    [
        {"name": "SQ-6001", "supplier": "A공급", "items": [{"item_code": "IT-800", "description": "강화선심, 미끄럼방지 밑창, 사이즈 260", "qty": 30, "rate": 15000}]},
        {"name": "SQ-6002", "supplier": "B공급", "items": [{"item_code": "IT-800", "description": "일반 선심(강화 아님), 미끄럼방지 밑창, 사이즈 260", "qty": 30, "rate": 12000}]},
        {"name": "SQ-6003", "supplier": "C공급", "items": [{"item_code": "IT-800", "description": "강화선심, 미끄럼방지 밑창, 사이즈 255/265만 있음(260 없음)", "qty": 30, "rate": 13000}]},
    ],
)
add_sq(
    "RFQ-7001",
    [{"item_code": "IT-500", "item_name": "복사용지", "description": "A4 사이즈, 80g", "qty": 200}],
    [
        {"name": "SQ-7001", "supplier": "가온문구", "items": [{"item_code": "IT-500", "description": "A4 사이즈, 80g", "qty": 200, "rate": 20000, "expected_delivery_date": "2026-09-12"}]},
        {"name": "SQ-7002", "supplier": "나은문구", "items": [{"item_code": "IT-500", "description": "A4 사이즈, 80g", "qty": 200, "rate": 19000, "expected_delivery_date": "2026-09-22"}]},
        {"name": "SQ-7003", "supplier": "다온문구", "items": [{"item_code": "IT-500", "description": "A4 사이즈, 75g 재생용지(요청 규격보다 낮음)", "qty": 200, "rate": 17000, "expected_delivery_date": "2026-09-15"}]},
    ],
)
add_sq(
    "RFQ-8001",
    [{"item_code": "IT-200", "item_name": "니트릴 코팅장갑", "description": "니트릴 코팅장갑 Cut Level D", "qty": 1000}],
    [
        {"name": "SQ-8001", "supplier": "D산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 Cut Level D", "qty": 500, "rate": 2500}]},
        {"name": "SQ-8002", "supplier": "E산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 Cut Level D", "qty": 1000, "rate": 3000}]},
        {"name": "SQ-8003", "supplier": "F산업", "items": [{"item_code": "IT-200", "description": "니트릴 코팅장갑 Cut Level C (요청보다 낮은 등급)", "qty": 1000, "rate": 2000}]},
    ],
)

from collections import Counter
print("태스크별 케이스 수:", Counter({k: len(v) for k, v in TEST_CASES.items()}))


In [ ]:
REFERENCES = {"item_spec_definition": [], "find_substitute": [], "sq_evaluation": []}

print("gpt-4o-mini로 정답(레퍼런스) 생성 중... (item_spec_definition / find_substitute / sq_evaluation)")

for case in TEST_CASES["item_spec_definition"]:
    prompt = build_prompt_item_spec_definition(case["input"]["item_group"])
    raw, _ = run_gpt(prompt)
    try:
        REFERENCES["item_spec_definition"].append(parse_json_response(raw))
    except Exception as e:
        print(f"  [경고] item_spec_definition 레퍼런스 파싱 실패 ({case['input']['item_group']}): {e}")
        REFERENCES["item_spec_definition"].append({"required_specs": [], "reason": ""})

for case in TEST_CASES["find_substitute"]:
    inp = case["input"]
    prompt = build_prompt_find_substitute(inp["item_name"], inp["item_description"], inp["qty_needed"], inp["candidates"])
    raw, _ = run_gpt(prompt)
    try:
        REFERENCES["find_substitute"].append(parse_json_response(raw))
    except Exception as e:
        print(f"  [경고] find_substitute 레퍼런스 파싱 실패 ({inp['item_name']}): {e}")
        REFERENCES["find_substitute"].append({"ranking": []})

for case in TEST_CASES["sq_evaluation"]:
    inp = case["input"]
    prompt = build_prompt_sq_evaluation(inp["requirements"], inp["quotations"])
    raw, _ = run_gpt(prompt)
    try:
        REFERENCES["sq_evaluation"].append(parse_json_response(raw))
    except Exception as e:
        print(f"  [경고] sq_evaluation 레퍼런스 파싱 실패 ({inp['requirements']['rfq_name']}): {e}")
        REFERENCES["sq_evaluation"].append({"ranking": []})

with open("/content/reference_answers.json", "w", encoding="utf-8") as f:
    json.dump(REFERENCES, f, ensure_ascii=False, indent=2)

print("완료 - /content/reference_answers.json 에 저장했으니 궁금하면 직접 열어서 검수 가능")


In [ ]:
def score_item_spec_validation(predicted: dict, expected: dict) -> dict:
    exp_by_spec = {r["spec"]: r["present"] for r in expected["results"]}
    pred_by_spec_cf = {}
    for r in predicted.get("results", []):
        spec = str(r.get("spec") or "").strip()
        if spec:
            pred_by_spec_cf[spec.casefold()] = bool(r.get("present") is True)
    correct = 0
    for spec, exp_present in exp_by_spec.items():
        pred_present = pred_by_spec_cf.get(spec.casefold(), False)
        if pred_present == exp_present:
            correct += 1
    return {"field_accuracy": correct / len(exp_by_spec) if exp_by_spec else 1.0}


def score_find_substitute(predicted: dict, reference: dict) -> dict:
    pred_codes = {r.get("item_code") for r in predicted.get("ranking", []) if r.get("item_code")}
    ref_codes = {r.get("item_code") for r in reference.get("ranking", []) if r.get("item_code")}
    if not ref_codes and not pred_codes:
        set_f1 = 1.0
    elif not ref_codes or not pred_codes:
        set_f1 = 0.0
    else:
        inter = pred_codes & ref_codes
        precision = len(inter) / len(pred_codes)
        recall = len(inter) / len(ref_codes)
        set_f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

    pred_top1 = next((r.get("item_code") for r in predicted.get("ranking", []) if r.get("rank") == 1), None)
    ref_top1 = next((r.get("item_code") for r in reference.get("ranking", []) if r.get("rank") == 1), None)
    if ref_top1 is None:
        top1_match = 1.0 if pred_top1 is None else 0.0
    else:
        top1_match = 1.0 if pred_top1 == ref_top1 else 0.0
    return {"set_f1": set_f1, "top1_match": top1_match}


def score_sq_evaluation(predicted: dict, reference: dict) -> dict:
    def _top1(ranking):
        return next((r.get("name") for r in ranking if r.get("rank") == 1), None)

    pred_ranking = predicted.get("ranking", [])
    ref_ranking = reference.get("ranking", [])
    pred_top1 = _top1(pred_ranking)
    ref_top1 = _top1(ref_ranking)
    top1_match = 1.0 if (ref_top1 is not None and pred_top1 == ref_top1) else 0.0

    pred_rank = {r.get("name"): r.get("rank") for r in pred_ranking if r.get("name")}
    ref_rank = {r.get("name"): r.get("rank") for r in ref_ranking if r.get("name")}
    common = [n for n in ref_rank if n in pred_rank]
    pairs = 0
    agree = 0
    for i in range(len(common)):
        for j in range(i + 1, len(common)):
            a, b = common[i], common[j]
            pairs += 1
            if (ref_rank[a] < ref_rank[b]) == (pred_rank[a] < pred_rank[b]):
                agree += 1
    pairwise_rank_agreement = (agree / pairs) if pairs else (1.0 if len(common) <= 1 else 0.0)
    return {"top1_match": top1_match, "pairwise_rank_agreement": pairwise_rank_agreement}


def score_item_spec_definition_judge(judge_result: dict, n_reference: int) -> dict:
    matched = judge_result.get("matched_reference_items", [])
    extra = judge_result.get("extra_candidate_items", [])
    n_matched = len(matched)
    recall = n_matched / n_reference if n_reference else 1.0
    denom = n_matched + len(extra)
    precision = n_matched / denom if denom else 1.0
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


TASK_METRICS = {
    "item_spec_validation": ["field_accuracy"],
    "item_spec_definition": ["precision", "recall", "f1"],
    "find_substitute": ["set_f1", "top1_match"],
    "sq_evaluation": ["top1_match", "pairwise_rank_agreement"],
}
PRIMARY_METRIC = {
    "item_spec_validation": "field_accuracy",
    "item_spec_definition": "f1",
    "find_substitute": "set_f1",
    "sq_evaluation": "top1_match",
}


In [ ]:
def _build_prompt(task_name, case):
    inp = case["input"]
    if task_name == "item_spec_validation":
        return build_prompt_item_spec_validation(inp["item_group"], inp["description"], inp["required_specs"])
    if task_name == "item_spec_definition":
        return build_prompt_item_spec_definition(inp["item_group"])
    if task_name == "find_substitute":
        return build_prompt_find_substitute(inp["item_name"], inp["item_description"], inp["qty_needed"], inp["candidates"])
    if task_name == "sq_evaluation":
        return build_prompt_sq_evaluation(inp["requirements"], inp["quotations"])
    raise ValueError(task_name)


def eval_case(task_name, case_idx, case, run_fn, model_name):
    prompt = _build_prompt(task_name, case)
    raw_text, elapsed = run_fn(prompt)
    row = {
        "model": model_name, "task": task_name, "case_id": case_idx,
        "elapsed_sec": elapsed, "raw_output": raw_text[:500],
    }
    try:
        parsed = parse_json_response(raw_text)
        row["parse_ok"] = True
    except Exception as e:
        row["parse_ok"] = False
        row["parse_error"] = str(e)
        for m in TASK_METRICS[task_name]:
            row[m] = float("nan")
        row["primary_score"] = float("nan")
        return row

    if task_name == "item_spec_validation":
        metrics = score_item_spec_validation(parsed, case["expected"])
    elif task_name == "item_spec_definition":
        ref = REFERENCES["item_spec_definition"][case_idx]
        judge_prompt = build_prompt_judge_spec_definition(
            case["input"]["item_group"], ref.get("required_specs", []), parsed.get("required_specs", []),
        )
        judge_raw, _ = run_gpt(judge_prompt)
        try:
            judge_parsed = parse_json_response(judge_raw)
        except Exception:
            judge_parsed = {
                "matched_reference_items": [],
                "extra_candidate_items": parsed.get("required_specs", []),
            }
        metrics = score_item_spec_definition_judge(judge_parsed, len(ref.get("required_specs", [])))
    elif task_name == "find_substitute":
        ref = REFERENCES["find_substitute"][case_idx]
        metrics = score_find_substitute(parsed, ref)
    elif task_name == "sq_evaluation":
        ref = REFERENCES["sq_evaluation"][case_idx]
        metrics = score_sq_evaluation(parsed, ref)

    row.update(metrics)
    row["primary_score"] = metrics[PRIMARY_METRIC[task_name]]
    return row


def eval_model(model_name, run_fn):
    rows = []
    for task_name, cases in TEST_CASES.items():
        for idx, case in enumerate(cases):
            print(f"  [{model_name}] {task_name} #{idx + 1}/{len(cases)}...")
            rows.append(eval_case(task_name, idx, case, run_fn, model_name))
    return rows


In [ ]:
all_rows = []

print("=== gpt-4o-mini 평가 (자기 자신을 레퍼런스로 채점하는 sanity check 포함) ===")
all_rows += eval_model("gpt-4o-mini", run_gpt)


In [ ]:
print("=== 텍스트 7B(4bit) 로딩 ===")
text_tok, text_model = load_text_model()

def run_text(prompt):
    return run_hf_text(text_tok, text_model, prompt)

all_rows += eval_model("text-7b-4bit", run_text)

del text_model, text_tok
free_gpu()


In [ ]:
print("=== 비전 7B(4bit, 텍스트 전용 모드) 로딩 ===")
vis_proc, vis_model = load_vision_model_text_only()

def run_vision_text(prompt):
    return run_hf_vision_text_only(vis_proc, vis_model, prompt)

all_rows += eval_model("vision-7b-text-mode-4bit", run_vision_text)

del vis_model, vis_proc
free_gpu()


In [ ]:
import pandas as pd

df = pd.DataFrame(all_rows)
df.to_csv("/content/eval_results_v3.csv", index=False)
print(f"총 {len(df)}행 저장됨 -> /content/eval_results_v3.csv")
df.head(10)


In [ ]:
# 태스크마다 채점 방식이 달라서 태스크별 "주 지표(primary_score)" 하나로 통일해 종합 비교
summary = df.groupby(["model", "task"]).agg(
    avg_primary_score=("primary_score", "mean"),
    parse_success_rate=("parse_ok", "mean"),
    avg_latency_sec=("elapsed_sec", "mean"),
    n=("case_id", "count"),
).reset_index()
summary


In [ ]:
overall = df.groupby("model").agg(
    avg_primary_score=("primary_score", "mean"),
    parse_success_rate=("parse_ok", "mean"),
    avg_latency_sec=("elapsed_sec", "mean"),
    n=("case_id", "count"),
).reset_index()
overall


In [ ]:
# 태스크별 세부 지표 (primary_score 하나로는 안 보이는 부분까지 확인)
for task, metrics in TASK_METRICS.items():
    metrics_label = ", ".join(metrics)
    print(f"\n--- {task} ({metrics_label}) ---")
    sub = df[df["task"] == task]
    display(sub.groupby("model")[metrics + ["elapsed_sec"]].mean())


## 결과 해석 가이드

- **gpt-4o-mini 행의 `item_spec_definition`/`find_substitute`/`sq_evaluation` 점수가
  1.0에서 많이 벗어나면** 레퍼런스 생성이나 채점 코드에 문제가 있다는 신호입니다
  (자기 자신과 비교하는 값이라 이론상 1.0에 가까워야 정상). 먼저 이걸 확인하세요.
- **`item_spec_validation`**은 사람이 직접 만든 설명 텍스트로 정답이 명확하게
  정해지는 유일한 태스크라, 이번 4개 태스크 중 가장 신뢰할 수 있는 지표입니다.
- **`item_spec_definition`**은 gpt-4o-mini 자신조차 "정답이 하나로 딱 떨어지지
  않는" 주관적 태스크라서, precision/recall/F1도 참고용이지 절대적인 기준으로
  보기는 어렵습니다. LLM 심사(judge)로 동의어 문제는 해결했지만, "심사위원도
  gpt-4o-mini"라는 한계는 남아있습니다.
- **`find_substitute`/`sq_evaluation`**은 item_code·견적문서명이라는 정확한
  식별자로 비교하기 때문에 동의어 문제가 없고, `top1_match`(실제 업무에서 가장
  중요한 "1등 선택이 맞았는가")를 중심으로 보는 게 가장 실무적으로 의미 있습니다.

## 한계

- 태스크당 8건, 총 32건 — 이전(5건)보다 늘었지만 여전히 통계적으로 확정적인
  결론을 내리기엔 작은 표본입니다.
- 테스트케이스는 실제 과거 데이터가 아니라 이번에 새로 만든 합성 케이스입니다.
- `item_spec_definition`/`find_substitute`/`sq_evaluation`의 "정답"은 실제
  정답이 아니라 "gpt-4o-mini라면 이렇게 답했을 것"이라는 기준입니다 — 즉 이
  평가는 "절대적으로 옳은가"가 아니라 "지금 쓰는 gpt-4o-mini를 얼마나 잘
  대체할 수 있는가"를 재는 것입니다. 목적에는 맞지만 혼동하지 않아야 합니다.
- gpt-4o-mini API는 temperature=0이어도 완전히 결정적이지 않을 수 있어
  (자기 자신과 비교한 점수가 1.0이 아니라 0.9x대로 나오는 정도는) 정상 범위로
  볼 수 있습니다.
- `vision-7b-text-mode-4bit`는 이미지 입력 없이 텍스트만 넣은 것으로, 이
  모델 본연의 용도(견적서 이미지 판독)는 이번 평가 범위가 아닙니다.
